# Homework Solutions: Lectures 7 & 8 - Confidence Intervals & Bootstrap

This notebook contains the Python code for the coding exercises (Problems 03, 04, 05).
See the PDF for full analytical solutions to all problems.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Armenian flag colors for plots
RED = '#D90012'
BLUE = '#0033A0'
ORANGE = '#F2A800'

---
## Problem 03: Bootstrap vs Formula

Generate $n = 30$ observations from $\text{Exp}(\lambda = 1)$ in Python (set `np.random.seed(509)`).

- a) Compute the **analytical** SE of $\bar{X}$ (since $\sigma = 1/\lambda = 1$, this is $\sigma / \sqrt{n}$).
- b) Compute the **bootstrap** SE with $B = 10{,}000$ resamples.
- c) Build three 95% CIs for $\mu = 1/\lambda$:
    - (i) Normal-theory: $\bar{X} \pm z_{0.025} \cdot \text{SE}$
    - (ii) Bootstrap percentile: $[\hat{\theta}^*_{0.025}, \hat{\theta}^*_{0.975}]$
    - (iii) $t$-interval: $\bar{X} \pm t_{n-1, 0.025} \cdot \frac{s}{\sqrt{n}}$

    Compare widths. Do all three contain the true $\mu = 1$?
- d) Repeat (a)-(c) for the **sample median** instead of the mean. Which method **cannot** give you a formula-based SE?

In [ ]:
np.random.seed(509)
n = 30
data = np.random.exponential(1.0, n)

xbar = np.mean(data)
s = np.std(data, ddof=1)
med = np.median(data)

print(f'Sample mean   = {xbar:.4f}')
print(f'Sample std    = {s:.4f}')
print(f'Sample median = {med:.4f}')

In [ ]:
# (a) Analytical SE
sigma = 1.0  # known for Exp(1)
se_analytical = sigma / np.sqrt(n)
print(f'Analytical SE(xbar) = {se_analytical:.4f}')

In [ ]:
# (b) Bootstrap SE
B = 10_000
boot_means = np.array([
    np.mean(np.random.choice(data, n, replace=True))
    for _ in range(B)
])
se_boot = np.std(boot_means, ddof=1)
print(f'Bootstrap SE  = {se_boot:.4f}')
print(f'Ratio boot/analytical = {se_boot / se_analytical:.3f}')

In [ ]:
# (c) Three 95% CIs
z_crit = 1.96
t_crit = stats.t.ppf(0.975, n - 1)

ci_normal = (xbar - z_crit * se_analytical, xbar + z_crit * se_analytical)
ci_boot = (np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5))
ci_t = (xbar - t_crit * s / np.sqrt(n), xbar + t_crit * s / np.sqrt(n))

print(f'Normal-theory CI: ({ci_normal[0]:.4f}, {ci_normal[1]:.4f})  width = {ci_normal[1]-ci_normal[0]:.4f}')
print(f'Bootstrap perc CI: ({ci_boot[0]:.4f}, {ci_boot[1]:.4f})  width = {ci_boot[1]-ci_boot[0]:.4f}')
print(f't-interval CI:    ({ci_t[0]:.4f}, {ci_t[1]:.4f})  width = {ci_t[1]-ci_t[0]:.4f}')
print()
print(f'True mu = 1')
print(f'  Normal contains mu=1? {ci_normal[0] < 1 < ci_normal[1]}')
print(f'  Boot   contains mu=1? {ci_boot[0] < 1 < ci_boot[1]}')
print(f'  t      contains mu=1? {ci_t[0] < 1 < ci_t[1]}')

In [ ]:
# Histogram of bootstrap means with the three CIs
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(boot_means, bins=60, density=True, color=BLUE, alpha=0.4,
        edgecolor='white', label='Bootstrap means')

# Mark CIs
y_pos = [0.3, 0.6, 0.9]
labels = ['Normal-theory', 'Bootstrap percentile', 't-interval']
colors = [RED, BLUE, ORANGE]
cis = [ci_normal, ci_boot, ci_t]

for ci, y, label, color in zip(cis, y_pos, labels, colors):
    ax.plot(ci, [y, y], 'o-', color=color, linewidth=2.5,
            markersize=8, label=f'{label}: ({ci[0]:.3f}, {ci[1]:.3f})')

ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='True $\mu = 1$')
ax.axvline(xbar, color='gray', linestyle=':', linewidth=1.5, label=f'$\\bar{{X}} = {xbar:.3f}$')

ax.set_xlabel('Sample mean')
ax.set_ylabel('Density')
ax.set_title('Problem 03: Bootstrap distribution of $\\bar{X}$ with three 95% CIs')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# (d) Bootstrap for the median
boot_medians = np.array([
    np.median(np.random.choice(data, n, replace=True))
    for _ in range(B)
])
se_boot_med = np.std(boot_medians, ddof=1)
ci_med = (np.percentile(boot_medians, 2.5), np.percentile(boot_medians, 97.5))
true_median = np.log(2)

print(f'Sample median = {med:.4f}')
print(f'Bootstrap SE(median) = {se_boot_med:.4f}')
print(f'Bootstrap 95% CI: ({ci_med[0]:.4f}, {ci_med[1]:.4f})')
print(f'True median of Exp(1) = ln(2) = {true_median:.4f}')
print(f'Contains true median? {ci_med[0] < true_median < ci_med[1]}')
print()
print('The normal-theory and t-interval CANNOT be applied to the median')
print('because there is no closed-form SE formula. Bootstrap is the way to go.')

In [ ]:
# Histogram of bootstrap medians
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(boot_medians, bins=60, density=True, color=ORANGE, alpha=0.4,
        edgecolor='white', label='Bootstrap medians')
ax.axvline(true_median, color='black', linestyle='--', linewidth=1.5,
           label=f'True median = ln(2) = {true_median:.3f}')
ax.axvline(med, color='gray', linestyle=':', linewidth=1.5,
           label=f'Sample median = {med:.3f}')

ax.plot(ci_med, [0.3, 0.3], 'o-', color=RED, linewidth=2.5,
        markersize=8, label=f'Bootstrap 95% CI: ({ci_med[0]:.3f}, {ci_med[1]:.3f})')

ax.set_xlabel('Sample median')
ax.set_ylabel('Density')
ax.set_title('Problem 03d: Bootstrap distribution of the median')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Problem 04: Bootstrap the Correlation

Given these $(x, y)$ pairs:
$$
(1,2),\; (2,3),\; (3,5),\; (4,4),\; (5,7),\; (6,8),\; (7,6),\; (8,9),\; (9,10),\; (10,12)
$$

- a) Compute the Pearson correlation $r$.
- b) Use the bootstrap ($B = 5{,}000$) to build a 95% percentile CI for the population correlation $\rho$.
- c) Plot the bootstrap distribution of $r^*$. Is it symmetric? If not, why might that be?
- d) Fisher's $z$-transform gives an analytical CI: transform $z = \tfrac{1}{2}\ln\tfrac{1+r}{1-r}$, build a normal CI using $\text{SE}(z) = \tfrac{1}{\sqrt{n-3}}$, then back-transform with $r = \tfrac{e^{2z}-1}{e^{2z}+1}$. Compare with your bootstrap CI.

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
y = np.array([2, 3, 5, 4, 7, 8, 6, 9, 10, 12])
n4 = len(x)

# (a) Pearson r
r = np.corrcoef(x, y)[0, 1]
print(f'Pearson r = {r:.4f}')

In [ ]:
# (b) Bootstrap CI for correlation
np.random.seed(509)
B4 = 5_000
boot_r = []
for _ in range(B4):
    idx = np.random.choice(n4, n4, replace=True)
    boot_r.append(np.corrcoef(x[idx], y[idx])[0, 1])

boot_r = np.array(boot_r)
boot_r = boot_r[~np.isnan(boot_r)]  # drop NaN from degenerate resamples

ci_boot_r = (np.percentile(boot_r, 2.5), np.percentile(boot_r, 97.5))
print(f'Bootstrap 95% CI for rho: ({ci_boot_r[0]:.4f}, {ci_boot_r[1]:.4f})')

In [ ]:
# (c) Plot bootstrap distribution - is it symmetric?
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(boot_r, bins=80, density=True, color=BLUE, alpha=0.4,
        edgecolor='white', label='Bootstrap $r^*$')
ax.axvline(r, color=RED, linestyle='--', linewidth=2,
           label=f'Sample r = {r:.4f}')
ax.axvline(ci_boot_r[0], color=ORANGE, linestyle=':', linewidth=1.5)
ax.axvline(ci_boot_r[1], color=ORANGE, linestyle=':', linewidth=1.5,
           label=f'95% CI: ({ci_boot_r[0]:.3f}, {ci_boot_r[1]:.3f})')

ax.set_xlabel('Bootstrap correlation $r^*$')
ax.set_ylabel('Density')
ax.set_title('Problem 04c: Bootstrap distribution of Pearson $r$ (strongly left-skewed!)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'Skewness = {stats.skew(boot_r):.2f}')
print('The distribution is strongly left-skewed because r is near its upper bound of 1.')
print('There is little room to go higher, but plenty of room to go lower.')

In [ ]:
# (d) Fisher z-transform CI
z_fish = 0.5 * np.log((1 + r) / (1 - r))
se_z = 1 / np.sqrt(n4 - 3)

z_lo = z_fish - 1.96 * se_z
z_hi = z_fish + 1.96 * se_z

# Back-transform
r_lo = (np.exp(2 * z_lo) - 1) / (np.exp(2 * z_lo) + 1)
r_hi = (np.exp(2 * z_hi) - 1) / (np.exp(2 * z_hi) + 1)

print(f'Fisher z = {z_fish:.4f}, SE(z) = {se_z:.4f}')
print(f'Fisher 95% CI for rho: ({r_lo:.4f}, {r_hi:.4f})')
print(f'Bootstrap 95% CI:      ({ci_boot_r[0]:.4f}, {ci_boot_r[1]:.4f})')
print()
print('Fisher CI is wider on the left (more conservative), because it properly')
print('symmetrizes the distribution via the z-transform.')

---
## Problem 05: When Bootstrap Breaks

Consider $X_1, \ldots, X_n \overset{\text{i.i.d.}}{\sim} \text{Uniform}(0, \theta)$. The MLE is $\hat{\theta} = X_{(n)} = \max(X_i)$.

- a) Show that $n(\theta - X_{(n)}) \xrightarrow{d} \text{Exp}(1/\theta)$. *(see PDF for proof)*
- b) Using $n = 50$ and $\theta = 1$, simulate 10,000 samples. For each sample, also run $B = 1{,}000$ bootstrap resamples of $\hat{\theta}^*$. Compare the true distribution of $n(\theta - \hat{\theta})$ with the bootstrap distribution of $n(\hat{\theta} - \hat{\theta}^*)$. Do they match?
- c) Explain why the bootstrap fails here. *(see PDF for explanation)*

In [ ]:
# (b) Simulation: true distribution vs bootstrap distribution
np.random.seed(509)
n5, theta = 50, 1.0
n_sim = 10_000
B5 = 1_000

true_dist = np.zeros(n_sim)
# For the bootstrap, we pick ONE representative sample and show its bootstrap dist
# Also collect all bootstrap values for aggregate comparison
boot_all = []

for i in range(n_sim):
    sample = np.random.uniform(0, theta, n5)
    theta_hat = sample.max()
    true_dist[i] = n5 * (theta - theta_hat)

    # Bootstrap only for first sample (for individual demo)
    if i == 0:
        boot_maxes_single = np.array([
            np.random.choice(sample, n5, replace=True).max()
            for _ in range(B5)
        ])
        boot_single = n5 * (theta_hat - boot_maxes_single)
        theta_hat_first = theta_hat

print(f'True distribution: mean = {true_dist.mean():.3f}, std = {true_dist.std():.3f}')
print(f'(Should be close to Exp(1): mean=1, std=1)')
print()
print(f'First sample: theta_hat = {theta_hat_first:.4f}')
print(f'Bootstrap dist: mean = {boot_single.mean():.3f}, std = {boot_single.std():.3f}')
print(f'Fraction of bootstrap samples where theta_hat* = theta_hat: '
      f'{np.mean(boot_single == 0):.3f}')
print(f'(Expected: ~1 - (1-1/n)^n = 1 - (49/50)^50 = {1-(49/50)**50:.3f})')

In [ ]:
# Plot: true vs bootstrap distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: true distribution with Exp(1) overlay
ax = axes[0]
ax.hist(true_dist, bins=60, density=True, color=BLUE, alpha=0.4,
        edgecolor='white', label='True: $n(\\theta - \\hat{\\theta})$')
t_grid = np.linspace(0, 8, 200)
ax.plot(t_grid, np.exp(-t_grid), color=RED, linewidth=2,
        label='Exp(1) density')
ax.set_xlabel('$n(\\theta - \\hat{\\theta})$')
ax.set_ylabel('Density')
ax.set_title('True sampling distribution')
ax.legend(fontsize=9)

# Right: bootstrap distribution (single sample)
ax = axes[1]
# Filter out the point mass at 0 for visibility
boot_nonzero = boot_single[boot_single > 0]
ax.hist(boot_nonzero, bins=40, density=False, color=ORANGE, alpha=0.4,
        edgecolor='white', label=f'Bootstrap (non-zero): {len(boot_nonzero)}/{B5}')
# Show the point mass
n_zero = np.sum(boot_single == 0)
ax.bar(0, n_zero, width=0.15, color=RED, alpha=0.7,
       label=f'Point mass at 0: {n_zero}/{B5} ({n_zero/B5:.1%})')
ax.set_xlabel('$n(\\hat{\\theta} - \\hat{\\theta}^*)$')
ax.set_ylabel('Count')
ax.set_title('Bootstrap distribution (one sample)')
ax.legend(fontsize=9)

fig.suptitle('Problem 05: Bootstrap FAILS for the uniform maximum',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('The true distribution matches Exp(1) perfectly.')
print('The bootstrap distribution has a huge point mass at 0 and does NOT match.')
print(f'\nReason: bootstrap always has theta_hat* <= theta_hat, so ~63% of resamples')
print('hit the boundary. The convergence rate is n (not sqrt(n)), which breaks the bootstrap.')